##### Band	Total Count:
##### many	169,452
##### medium 14,434
##### few	3,698

In [ ]:
import pandas as pd
import os
import glob

def analyze_sideslip_files(distribution_csv, folder_path="."):
    """
    Analyzes all CSVs in a folder and categorizes sideSlip values into 
    few, medium, and many based on a distribution reference file.
    """
    # 1. Load the reference distribution mapping
    dist_df = pd.read_csv(distribution_csv)
    
    # Create a mapping dictionary: {sideSlip_value: band_name}
    # We round keys to 1 decimal place to ensure matching with typical sensor data
    mapping = dict(zip(dist_df['sideSlip'].round(1), dist_df['band']))
    
    # 2. Identify your 22 training CSV files
    # This looks for all .csv files except the distribution file itself
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    train_files = [f for f in all_files if os.path.basename(f) != distribution_csv]
    
    print(f"Found {len(train_files)} files to analyze.\n")
    
    file_summaries = []

    # 3. Process each training file
    for file_path in train_files:
        try:
            df = pd.read_csv(file_path)
            file_name = os.path.basename(file_path)
            
            if 'sideSlip' not in df.columns:
                print(f"Skipping {file_name}: No 'sideSlip' column found.")
                continue
            
            # Map the sideSlip values in this file to the bands
            # We round to 1 decimal to match our mapping dictionary keys
            df['detected_band'] = df['sideSlip'].round(1).map(mapping)
            
            # Count the occurrences of each band
            counts = df['detected_band'].value_counts()
            
            # Save the results for this file
            file_summaries.append({
                'File Name': file_name,
                'Few': counts.get('few', 0),
                'Medium': counts.get('medium', 0),
                'Many': counts.get('many', 0),
                'Total Rows': len(df)
            })
            
        except Exception as e:
            print(f"Error reading {file_name}: {e}")

    # 4. Create a final summary table
    summary_df = pd.DataFrame(file_summaries)
    
    # Calculate percentages for deeper EDA
    summary_df['Few %'] = (summary_df['Few'] / summary_df['Total Rows'] * 100).round(2)
    summary_df['Medium %'] = (summary_df['Medium'] / summary_df['Total Rows'] * 100).round(2)
    summary_df['Many %'] = (summary_df['Many'] / summary_df['Total Rows'] * 100).round(2)
    
    return summary_df

# --- Execution ---
# Replace '.' with the path to your folder if the files are not in the same directory
results = analyze_sideslip_files('../datasets_challenge/sideSlip_distribution.csv', folder_path="../../datasets_challenge/train")

# Display the result
print(results.to_string(index=False))

# Optional: Save the summary to a new CSV
results.to_csv('sideslip_analysis_summary.csv', index=False)